<a href="https://colab.research.google.com/github/phoenixcapera04/Dissertation-R-Shiny/blob/main/Final_Synthetic_Data_b.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>


# Final_Synthetic_Data — Reproducible Synthetic Dataset for Shiny Wellbeing Dashboard

**Author:** _Generated via assistant_  
**Runtime:** Google Colab (Python 3)  
**Outputs:** CSVs saved to `/content/data/`

This notebook builds a fully reproducible, modular synthetic dataset for a school wellbeing dashboard.  
Each major section starts with markdown explaining the **goal, logic, and parameters**, followed by the code that **generates a single table**, previews it, and saves it to CSV.

## Contents
1. Setup & Config
2. Academic Calendars
   - 2.1 `academic_years_df`
   - 2.2 `academic_weeks_df` (39 weeks/year, excluding summer)
   - 2.3 `term_days_df` (optional: 5 days/week derived from weeks)
3. Students & Progression
   - 3.1 `students_df` (student population by year, with progression)
4. WHO-5 Weekly Wellbeing
   - 4.1 `who5_df` (weekly items q1–q5 and derived metrics)
5. Weekly Engagement
   - 5.1 `engagement_df` (correlated with WHO-5)
6. Safeguarding Flags
   - 6.1 `safeguarding_flags_df` (static & dynamic flags)
7. Master Views
   - 7.1 `master_weekly`
   - 7.2 `master_yearly`
8. EDA Columns (per student-year)
   - WHO-5 skewness, kurtosis, volatility
   - Engagement volatility
   - WHO-5↔Engagement correlation
   - Flag prevalence rates per cohort/year
9. Save all CSVs & Quick Summaries



## 1) Setup & Config

- Uses `numpy.random.seed()` for reproducibility.
- Output directory: `/content/data/` (created if not exists).
- Academic years: 2019–2020 to 2024–2025.
- Weeks per academic year: **39** (approx. real school weeks, excluding summer).
- Cohorts: **Years 7–12**; students move up 1 per year; graduate after Year 12.
- Students per cohort-year: 45 (3 sub-cohorts: A/B/C, 15 each).


In [1]:

# !pip install pandas numpy scipy tqdm --quiet

import os
import math
import uuid
import numpy as np
import pandas as pd
from scipy.stats import skew, kurtosis, pearsonr
from datetime import date, timedelta

# Reproducibility
np.random.seed(42)

# Output directory for Colab
OUTPUT_DIR = "/content/data/"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Global Config
AY_START = 2019   # academic year starting in Sep 2019 (2019-2020)
AY_END   = 2024   # academic year starting in Sep 2024 (2024-2025)
WEEKS_PER_YEAR = 39
COHORTS = list(range(7, 13))  # Years 7..12
SUB_COHORTS = ["A", "B", "C"]
STUDENTS_PER_SUB = 15  # 15 per sub cohort -> 45 per cohort-year
STUDENTS_PER_COHORT = STUDENTS_PER_SUB * len(SUB_COHORTS)

# Helper: academic year label and start date (1st Monday of September)
def academic_year_label(year_start):
    return f"{year_start}-{year_start+1}"

def first_monday_of_september(year):
    # Find first Monday in September for a given calendar year
    d = date(year, 9, 1)
    while d.weekday() != 0:  # 0 = Monday
        d += timedelta(days=1)
    return d



## 2) Academic Calendars

We create **academic years**, **weeks**, and (optionally) **term days** derived from weeks.
- **Years:** Labels like `"2019-2020"` and a `year_start` date (first Monday of September).
- **Weeks:** Exactly **39** school weeks per academic year (`week_of_year = 1..39`), no summer weeks.
- **Term Days:** 5 days per school week (Mon–Fri), date-stamped from each week's Monday.


### 2.1 `academic_years_df`

In [3]:
years = []
for y in range(AY_START, AY_END + 1):
    years.append({
        "academic_year": academic_year_label(y),
        "year_start": first_monday_of_september(y)
    })
academic_years_df = pd.DataFrame(years)

display(academic_years_df.head())
print(academic_years_df.info())
print(academic_years_df.describe())

academic_years_df.to_csv(os.path.join(OUTPUT_DIR, "academic_years_df.csv"), index=False)

,academic_year,year_start
0,2019-2020,2019-09-02
1,2020-2021,2020-09-07
2,2021-2022,2021-09-06
3,2022-2023,2022-09-05
4,2023-2024,2023-09-04


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6 entries, 0 to 5
Data columns (total 2 columns):
 #   Column         Non-Null Count  Dtype 
---  ------         --------------  ----- 
 0   academic_year  6 non-null      object
 1   year_start     6 non-null      object
dtypes: object(2)
memory usage: 228.0+ bytes
None
       academic_year  year_start
count              6           6
unique             6           6
top        2019-2020  2019-09-02
freq               1           1


### 2.2 `academic_weeks_df` — 39 school weeks per year

In [7]:
weeks = []
for y in range(AY_START, AY_END + 1):
    ay = academic_year_label(y)
    start_monday = first_monday_of_september(y)
    for w in range(1, WEEKS_PER_YEAR + 1):
        week_monday = start_monday + timedelta(weeks=w-1)
        weeks.append({
            "week_id": f"{ay}-W{w:02d}",
            "academic_year": ay,
            "week_of_year": w,
            "week_start": week_monday
        })
academic_weeks_df = pd.DataFrame(weeks)

display(academic_weeks_df.head())
print(academic_weeks_df.info())
print(academic_weeks_df.describe())

academic_weeks_df.to_csv(os.path.join(OUTPUT_DIR, "academic_weeks_df.csv"), index=False)

,week_id,academic_year,week_of_year,week_start
0,2019-2020-W01,2019-2020,1,2019-09-02
1,2019-2020-W02,2019-2020,2,2019-09-09
2,2019-2020-W03,2019-2020,3,2019-09-16
3,2019-2020-W04,2019-2020,4,2019-09-23
4,2019-2020-W05,2019-2020,5,2019-09-30


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 234 entries, 0 to 233
Data columns (total 4 columns):
 #   Column         Non-Null Count  Dtype 
---  ------         --------------  ----- 
 0   week_id        234 non-null    object
 1   academic_year  234 non-null    object
 2   week_of_year   234 non-null    int64 
 3   week_start     234 non-null    object
dtypes: int64(1), object(3)
memory usage: 7.4+ KB
None
       week_of_year
count    234.000000
mean      20.000000
std       11.278754
min        1.000000
25%       10.000000
50%       20.000000
75%       30.000000
max       39.000000



### 2.3 `term_days_df` (optional)

We derive **school days** (Mon–Fri) for each academic week to support daily granularity if needed.


In [9]:
term_days = []
for _, row in academic_weeks_df.iterrows():
    week_start = row["week_start"]
    for d in range(5):  # Mon..Fri
        term_days.append({
            "week_id": row["week_id"],
            "academic_year": row["academic_year"],
            "week_of_year": row["week_of_year"],
            "date": week_start + timedelta(days=d),
            "day_of_week": (week_start + timedelta(days=d)).strftime("%A")
        })
term_days_df = pd.DataFrame(term_days)

display(term_days_df.head())
print(term_days_df.info())
print(term_days_df.describe())

term_days_df.to_csv(os.path.join(OUTPUT_DIR, "term_days_df.csv"), index=False)

,week_id,academic_year,week_of_year,date,day_of_week
0,2019-2020-W01,2019-2020,1,2019-09-02,Monday
1,2019-2020-W01,2019-2020,1,2019-09-03,Tuesday
2,2019-2020-W01,2019-2020,1,2019-09-04,Wednesday
3,2019-2020-W01,2019-2020,1,2019-09-05,Thursday
4,2019-2020-W01,2019-2020,1,2019-09-06,Friday


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1170 entries, 0 to 1169
Data columns (total 5 columns):
 #   Column         Non-Null Count  Dtype 
---  ------         --------------  ----- 
 0   week_id        1170 non-null   object
 1   academic_year  1170 non-null   object
 2   week_of_year   1170 non-null   int64 
 3   date           1170 non-null   object
 4   day_of_week    1170 non-null   object
dtypes: int64(1), object(4)
memory usage: 45.8+ KB
None
       week_of_year
count   1170.000000
mean      20.000000
std       11.259441
min        1.000000
25%       10.000000
50%       20.000000
75%       30.000000
max       39.000000



## 3) Students & Progression — `students_df`

**Goal:** Build a **student population table** with cohort progression across academic years.

**Logic:**
- Each academic year creates a new Year 7 **intake** of 45 students (A/B/C × 15).
- Students **move up one cohort** per year until Year 12, then **graduate**.
- Each row represents a **student-year**, with status `active` if the student is in Years 7–12 that year.
- `start_cohort` is the cohort they first entered (always 7 for new intakes here).

**Columns:**
- `student_id` (UUID, persistent across years)
- `academic_year`, `cohort` (current year group), `start_cohort`
- `sub_cohort` (A/B/C)
- `status` (`active` / `not_active`)


In [11]:

# Create base student registry: entering Year 7 each academic year
intake_registry = []  # one row per new student at the year of entry
for y in range(AY_START, AY_END + 1):
    for sub in SUB_COHORTS:
        for i in range(STUDENTS_PER_SUB):
            intake_registry.append({
                "student_id": str(uuid.uuid4()),
                "entry_year": academic_year_label(y),
                "sub_cohort": sub,
                "start_cohort": 7  # always enter at Year 7 in this model
            })
intake_df = pd.DataFrame(intake_registry)

# Expand to student-year records with progression
student_year_rows = []
for _, s in intake_df.iterrows():
    # Student starts in Year 7 at entry_year
    entry_start = int(s["entry_year"].split("-")[0])
    # They can be present from entry_year until they would pass Year 12
    for y in range(entry_start, AY_END + 1):
        cohort = 7 + (y - entry_start)
        status = "active" if (7 <= cohort <= 12) else "not_active"
        student_year_rows.append({
            "student_id": s["student_id"],
            "academic_year": academic_year_label(y),
            "cohort": cohort,
            "start_cohort": s["start_cohort"],
            "sub_cohort": s["sub_cohort"],
            "status": status
        })
students_df = pd.DataFrame(student_year_rows)

# Keep only rows within 7..12; others are post-graduation (set as not_active)
students_df.loc[(students_df["cohort"] < 7) | (students_df["cohort"] > 12), "status"] = "not_active"

display(students_df.head())
print(students_df.info())
print(students_df.describe())

students_df.to_csv(os.path.join(OUTPUT_DIR, "students_df.csv"), index=False)


,student_id,academic_year,cohort,start_cohort,sub_cohort,status
0,2af83920-568b-48fc-8d93-1bd1b1d54f4a,2019-2020,7,7,A,active
1,2af83920-568b-48fc-8d93-1bd1b1d54f4a,2020-2021,8,7,A,active
2,2af83920-568b-48fc-8d93-1bd1b1d54f4a,2021-2022,9,7,A,active
3,2af83920-568b-48fc-8d93-1bd1b1d54f4a,2022-2023,10,7,A,active
4,2af83920-568b-48fc-8d93-1bd1b1d54f4a,2023-2024,11,7,A,active


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 945 entries, 0 to 944
Data columns (total 6 columns):
 #   Column         Non-Null Count  Dtype 
---  ------         --------------  ----- 
 0   student_id     945 non-null    object
 1   academic_year  945 non-null    object
 2   cohort         945 non-null    int64 
 3   start_cohort   945 non-null    int64 
 4   sub_cohort     945 non-null    object
 5   status         945 non-null    object
dtypes: int64(2), object(4)
memory usage: 44.4+ KB
None
           cohort  start_cohort
count  945.000000         945.0
mean     8.666667           7.0
std      1.491501           0.0
min      7.000000           7.0
25%      7.000000           7.0
50%      8.000000           7.0
75%     10.000000           7.0
max     12.000000           7.0



## 4) WHO-5 Weekly Wellbeing — `who5_df`

**Goal:** Generate weekly WHO-5 scores **only for active students** in their study years.

**Specification:**
- 5 Likert items `q1..q5` in **[0, 5]**.
- `who5_total` in **[0, 100]**: sum(q1..q5)×4.
- Distribution target: **Mean ≈ 60**, **SD ≈ 20**.
- Derived:
  - `delta_1w` (week-to-week change)
  - `rolling_mean_4w` (4-week rolling mean)
  - `persist_low`: `True` if `<40` for **≥2 consecutive weeks**
- **Constraint:** Ensure **10–15%** of **student-years** have a persistent low streak.

**Approach:**
- Sample a **latent wellbeing** trajectory per student-year, then produce item-level responses.
- Post-process to **inject low streaks** where needed to hit target prevalence.


In [12]:

# Build active student-week frame by merging students with academic_weeks
active_students = students_df[students_df["status"] == "active"].copy()
active_sw = active_students.merge(academic_weeks_df, on="academic_year", how="inner")

def simulate_who5_for_group(g):
    # g contains rows for a single student within a single academic year
    n = len(g)
    # Latent base around 60 +/- 20 with slight weekly AR(1)-like drift
    base = 60 + np.random.normal(0, 20)
    noise = np.random.normal(0, 8, size=n)
    # Add gentle drift
    drift = np.cumsum(np.random.normal(0, 1.2, size=n))
    latent = base + noise + 0.6*drift
    # Clip latent to [5,95] to avoid extremes
    latent = np.clip(latent, 5, 95)

    # Convert latent to item means in [0,5], then sample integers 0..5
    item_mu = latent / 20.0  # 60 -> 3.0
    q = {}
    for i in range(1,6):
        q_i = np.clip(np.round(np.random.normal(item_mu, 1.2)), 0, 5).astype(int)
        q[f"q{i}"] = q_i

    df = g.copy()
    for k, v in q.items():
        df[k] = v
    df["who5_total"] = df[[f"q{i}" for i in range(1,6)]].sum(axis=1) * 4

    # Derived
    df = df.sort_values("week_of_year")
    df["delta_1w"] = df["who5_total"].diff()
    df["rolling_mean_4w"] = df["who5_total"].rolling(window=4, min_periods=1).mean()

    # Persist low: <40 for >= 2 consecutive
    below = df["who5_total"] < 40
    # find any run length >=2
    count = 0
    has_streak = False
    for flag in below:
        if flag:
            count += 1
            if count >= 2:
                has_streak = True
        else:
            count = 0
    df["persist_low"] = has_streak
    return df

# Simulate initial WHO-5
who5_df = active_sw.groupby(["student_id", "academic_year"], group_keys=False).apply(simulate_who5_for_group).reset_index(drop=True)

# Ensure 10–15% student-years have persistent low streaks by injecting dips where absent
sy = who5_df.groupby(["student_id","academic_year"])["persist_low"].first().reset_index()
current_rate = sy["persist_low"].mean()

target_low = np.random.uniform(0.10, 0.15)
if current_rate < target_low:
    # Select some student-years to inject low streaks
    needed = int((target_low - current_rate) * len(sy))
    candidates = sy[~sy["persist_low"]].sample(n=max(0, needed), random_state=42)
    for _, row in candidates.iterrows():
        mask = (who5_df["student_id"]==row["student_id"]) & (who5_df["academic_year"]==row["academic_year"])
        df_ = who5_df.loc[mask].sort_values("week_of_year").copy()
        if len(df_) >= 6:
            # pick a window of 2-3 weeks and set low values (random 25–35)
            start_idx = np.random.randint(1, len(df_)-2)
            window = np.random.randint(2, 4)
            idx = df_.index[start_idx:start_idx+window]
            who5_df.loc[idx, "who5_total"] = np.random.randint(25, 36, size=len(idx))
            # Recompute derived
            df_2 = who5_df.loc[mask].sort_values("week_of_year").copy()
            df_2["delta_1w"] = df_2["who5_total"].diff()
            df_2["rolling_mean_4w"] = df_2["who5_total"].rolling(window=4, min_periods=1).mean()
            below = df_2["who5_total"] < 40
            count = streak = 0
            has_streak = False
            for flag in below:
                if flag:
                    count += 1
                    if count >= 2: has_streak = True
                else:
                    count = 0
            who5_df.loc[mask, ["delta_1w","rolling_mean_4w"]] = df_2[["delta_1w","rolling_mean_4w"]].values
            who5_df.loc[mask, "persist_low"] = has_streak

display(who5_df.head())
print(who5_df.info())
print(who5_df.describe())
who5_df.to_csv(os.path.join(OUTPUT_DIR, "who5_df.csv"), index=False)


/tmp/ipython-input-3800942085.py:50: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  who5_df = active_sw.groupby(["student_id", "academic_year"], group_keys=False).apply(simulate_who5_for_group).reset_index(drop=True)


,student_id,academic_year,cohort,start_cohort,sub_cohort,status,week_id,week_of_year,week_start,q1,q2,q3,q4,q5,who5_total,delta_1w,rolling_mean_4w,persist_low
0,2af83920-568b-48fc-8d93-1bd1b1d54f4a,2019-2020,7,7,A,active,2019-2020-W01,1,2019-09-02,1,3,2,2,1,36,NaN,36.0,True
1,2af83920-568b-48fc-8d93-1bd1b1d54f4a,2019-2020,7,7,A,active,2019-2020-W02,2,2019-09-09,2,4,2,1,3,48,12.0,42.0,True
2,2af83920-568b-48fc-8d93-1bd1b1d54f4a,2019-2020,7,7,A,active,2019-2020-W03,3,2019-09-16,5,2,1,3,1,48,0.0,44.0,True
3,2af83920-568b-48fc-8d93-1bd1b1d54f4a,2019-2020,7,7,A,active,2019-2020-W04,4,2019-09-23,1,3,3,1,2,40,-8.0,43.0,True
4,2af83920-568b-48fc-8d93-1bd1b1d54f4a,2019-2020,7,7,A,active,2019-2020-W05,5,2019-09-30,4,3,2,0,5,56,16.0,48.0,True


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 36855 entries, 0 to 36854
Data columns (total 18 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   student_id       36855 non-null  object 
 1   academic_year    36855 non-null  object 
 2   cohort           36855 non-null  int64  
 3   start_cohort     36855 non-null  int64  
 4   sub_cohort       36855 non-null  object 
 5   status           36855 non-null  object 
 6   week_id          36855 non-null  object 
 7   week_of_year     36855 non-null  int64  
 8   week_start       36855 non-null  object 
 9   q1               36855 non-null  int64  
 10  q2               36855 non-null  int64  
 11  q3               36855 non-null  int64  
 12  q4               36855 non-null  int64  
 13  q5               36855 non-null  int64  
 14  who5_total       36855 non-null  int64  
 15  delta_1w         35910 non-null  float64
 16  rolling_mean_4w  36855 non-null  float64
 17  persist_low 


## 5) Weekly Engagement — `engagement_df`

**Goal:** Generate weekly engagement metrics **only for active students**, correlated with WHO-5.

**Specification:**
- Counts:
  - `logins` (0–5), `journal_entries` (0–5), `goals_set` (0–10)
- `total_engagement_score` (0–100)
- `no_engagement_2w` if **two or more consecutive** weeks with zero engagement

**Correlation:** target Pearson **0.4–0.6** with `who5_total` (per year).  
We generate a **latent engagement z** as a linear combination of standardized WHO-5 and white noise, then scale to [0,100].


In [13]:

# Merge base with WHO-5
ws = who5_df[["student_id","academic_year","week_id","week_of_year","who5_total"]].copy()

def simulate_engagement_for_group(g, rho=0.5):
    n = len(g)
    w_scaled = (g["who5_total"] - g["who5_total"].mean()) / (g["who5_total"].std(ddof=0) + 1e-6)
    eps = np.random.normal(0, 1, size=n)
    z = rho * w_scaled + np.sqrt(1 - rho**2) * eps
    # Scale to 0..100
    e_score = 50 + 15*z  # initial
    e_score = 100 * (e_score - e_score.min()) / (e_score.max() - e_score.min() + 1e-6)
    e_score = np.clip(e_score, 0, 100)

    df = g.copy()
    df["total_engagement_score"] = e_score

    # Map to counts with noise
    # 0-20 -> 0, 20-40 -> 1-2, 40-60 -> 2-3, 60-80 -> 3-4, 80-100 -> 4-5
    bins = pd.cut(df["total_engagement_score"], bins=[-1,20,40,60,80,101], labels=[0,1,2,3,4])
    base = bins.astype(int).values
    logins = np.clip(base + np.random.binomial(1, 0.3, size=n), 0, 5)
    journals = np.clip(base + np.random.binomial(1, 0.25, size=n), 0, 5)
    goals = np.clip((base+1) * 2 + np.random.randint(-1,2,size=n), 0, 10)

    df["logins"] = logins
    df["journal_entries"] = journals
    df["goals_set"] = goals

    # no_engagement_2w: any 2+ consecutive weeks with zero across all three
    zero_eng = ((df["logins"]==0) & (df["journal_entries"]==0) & (df["goals_set"]==0))
    count = 0
    has_two = False
    for zf in zero_eng:
        if zf:
            count += 1
            if count >= 2: has_two = True
        else:
            count = 0
    df["no_engagement_2w"] = has_two
    return df

engagement_df = ws.groupby(["student_id","academic_year"], group_keys=False).apply(simulate_engagement_for_group).reset_index(drop=True)

display(engagement_df.head())
print(engagement_df.info())
print(engagement_df.describe())

engagement_df.to_csv(os.path.join(OUTPUT_DIR, "engagement_df.csv"), index=False)


/tmp/ipython-input-677000784.py:42: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  engagement_df = ws.groupby(["student_id","academic_year"], group_keys=False).apply(simulate_engagement_for_group).reset_index(drop=True)


,student_id,academic_year,week_id,week_of_year,who5_total,total_engagement_score,logins,journal_entries,goals_set,no_engagement_2w
0,2af83920-568b-48fc-8d93-1bd1b1d54f4a,2019-2020,2019-2020-W01,1,36,24.412429,1,1,3,False
1,2af83920-568b-48fc-8d93-1bd1b1d54f4a,2019-2020,2019-2020-W02,2,48,50.876019,2,3,6,False
2,2af83920-568b-48fc-8d93-1bd1b1d54f4a,2019-2020,2019-2020-W03,3,48,66.172879,4,3,9,False
3,2af83920-568b-48fc-8d93-1bd1b1d54f4a,2019-2020,2019-2020-W04,4,40,53.854785,3,2,5,False
4,2af83920-568b-48fc-8d93-1bd1b1d54f4a,2019-2020,2019-2020-W05,5,56,99.999998,5,4,10,False


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 36855 entries, 0 to 36854
Data columns (total 10 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   student_id              36855 non-null  object 
 1   academic_year           36855 non-null  object 
 2   week_id                 36855 non-null  object 
 3   week_of_year            36855 non-null  int64  
 4   who5_total              36855 non-null  int64  
 5   total_engagement_score  36855 non-null  float64
 6   logins                  36855 non-null  int64  
 7   journal_entries         36855 non-null  int64  
 8   goals_set               36855 non-null  int64  
 9   no_engagement_2w        36855 non-null  bool   
dtypes: bool(1), float64(1), int64(5), object(3)
memory usage: 2.6+ MB
None
       week_of_year    who5_total  total_engagement_score        logins  \
count  36855.000000  36855.000000            36855.000000  36855.000000   
mean      20.000000     59.03


## 6) Safeguarding Flags — `safeguarding_flags_df`

**Goal:** Generate **static** (persist across years) and **dynamic** (vary by year) flags.

- **Static flags examples** (assigned once per student, persist): `EAL`, `Adopted`, `ASC Diagnosis`, `ADHD Diagnosis`, `SEND`, `Young Carer`, `EHCP`, `Pupil Premium`, `Refugee/Asylum Seeker`, `Looked After Child (LAC)`.
- **Dynamic flags examples** (assigned per student-year): `Attendance < 90%`, `Low Mood`, `Bereavement`, `Bullying`, `Social Worker Involved`, `Safeguarding Referral`, `Exclusion`, `CAMHS Referral`, `New Diagnosis`, `Family Difficulty`.

> **Note:** Replace/extend the lists below with your **full 45+ flags**. The code will adapt automatically.


In [14]:

STATIC_FLAGS = [
    "EAL","Adopted","ASC Diagnosis","ADHD Diagnosis","SEND","Young Carer","EHCP",
    "Pupil Premium","Refugee/Asylum Seeker","Looked After Child (LAC)",
    "Chronic Illness","Hearing Impairment","Visual Impairment","Dyslexia",
    "Dyspraxia","Autism Spectrum (ASC)","Speech/Language Need","Gifted & Talented",
    "Transport Assistance","Free School Meals"
]

DYNAMIC_FLAGS = [
    "Attendance < 90%","Low Mood","Bereavement","Bullying","Social Worker Involved",
    "Safeguarding Referral","Exclusion","CAMHS Referral","New Diagnosis",
    "Family Difficulty","Housing Instability","Child in Need Plan","CP Plan",
    "Police Involvement","Recent Move/Transition","Attendance < 80%",
    "Self-Harm Concern","Peer Conflict","Online Safety Concern","Substance Concern",
    "Parental Separation","Domestic Abuse Exposure","Financial Hardship",
    "Medical Appointment Burden","Young Carer Increased Duties"
]

# Assign static flags per student with low base rates
student_list = students_df[["student_id"]].drop_duplicates().copy()
static_rows = []
for _, row in student_list.iterrows():
    sid = row["student_id"]
    for f in STATIC_FLAGS:
        # Independent Bernoulli with small p (0.5%..8%)
        p = np.random.uniform(0.005, 0.08)
        if np.random.rand() < p:
            static_rows.append({"student_id": sid, "flag_name": f, "flag_type": "static"})
static_flags_df = pd.DataFrame(static_rows)

# Assign dynamic flags per student-year with context-sensitive probabilities
dynamic_rows = []
for _, sy in students_df.iterrows():
    sid = sy["student_id"]
    ay = sy["academic_year"]
    cohort = sy["cohort"]
    status = sy["status"]
    if status != "active":
        continue
    # Baseline p for dynamic flags (0.5%..12%)
    for f in DYNAMIC_FLAGS:
        base_p = np.random.uniform(0.01, 0.12)
        # Slightly higher chance of "Low Mood" if WHO-5 had persist_low
        if f in ["Low Mood", "CAMHS Referral", "Safeguarding Referral"]:
            # Check persist_low in who5
            mask = (who5_df["student_id"]==sid) & (who5_df["academic_year"]==ay)
            pl = False
            if mask.any():
                pl = who5_df.loc[mask, "persist_low"].iloc[0]
            if pl:
                base_p = min(0.35, base_p + 0.18)
        if np.random.rand() < base_p:
            dynamic_rows.append({
                "student_id": sid, "academic_year": ay, "flag_name": f, "flag_type": "dynamic"
            })
dynamic_flags_df = pd.DataFrame(dynamic_rows)

# Union table with academic_year for static (so it can merge yearly)
if static_flags_df.empty:
    safeguarding_flags_df = dynamic_flags_df.copy()
else:
    # Cross static flags with all years present for the student
    years_map = students_df.groupby("student_id")["academic_year"].unique()
    static_expanded = []
    for sid, years in years_map.items():
        sf = static_flags_df[static_flags_df["student_id"]==sid]
        for _, srow in sf.iterrows():
            for ay in years:
                static_expanded.append({
                    "student_id": sid, "academic_year": ay,
                    "flag_name": srow["flag_name"], "flag_type": "static"
                })
    static_expanded_df = pd.DataFrame(static_expanded)
    safeguarding_flags_df = pd.concat([static_expanded_df, dynamic_flags_df], ignore_index=True)

display(safeguarding_flags_df.head())
print(safeguarding_flags_df.info())
print(safeguarding_flags_df.describe())

safeguarding_flags_df.to_csv(os.path.join(OUTPUT_DIR, "safeguarding_flags_df.csv"), index=False)


,student_id,academic_year,flag_name,flag_type
0,0025c628-745e-4922-be94-a7d67f6d8177,2019-2020,Dyslexia,static
1,0025c628-745e-4922-be94-a7d67f6d8177,2020-2021,Dyslexia,static
2,0025c628-745e-4922-be94-a7d67f6d8177,2021-2022,Dyslexia,static
3,0025c628-745e-4922-be94-a7d67f6d8177,2022-2023,Dyslexia,static
4,0025c628-745e-4922-be94-a7d67f6d8177,2023-2024,Dyslexia,static


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2526 entries, 0 to 2525
Data columns (total 4 columns):
 #   Column         Non-Null Count  Dtype 
---  ------         --------------  ----- 
 0   student_id     2526 non-null   object
 1   academic_year  2526 non-null   object
 2   flag_name      2526 non-null   object
 3   flag_type      2526 non-null   object
dtypes: object(4)
memory usage: 79.1+ KB
None
                                  student_id academic_year  \
count                                   2526          2526   
unique                                   265             6   
top     a2b0a7f2-779b-4187-9f62-d8505b8d85ff     2024-2025   
freq                                      27           704   

                    flag_name flag_type  
count                    2526      2526  
unique                     45         2  
top     Safeguarding Referral   dynamic  
freq                      126      1713  



## 7) Master Views

We create two master views:

### 7.1 `master_weekly`
- Merge **students** (active only) + **academic_weeks** + **WHO-5** + **engagement**
- Join **safeguarding flags** for the same year, pivoted to counts:
  - `static_flag_count`, `dynamic_flag_count`, `total_flag_count`

### 7.2 `master_yearly`
- Per student-year aggregates:
  - WHO-5 mean & SD
  - Engagement mean & SD
  - Flag counts


In [15]:

# Weekly master
active_students = students_df[students_df["status"]=="active"].copy()

# Pivot flag counts per student-year
flag_counts = safeguarding_flags_df.groupby(["student_id","academic_year","flag_type"])["flag_name"].count().unstack(fill_value=0)
flag_counts = flag_counts.rename(columns=lambda c: f"{c}_flag_count")
for col in ["static_flag_count","dynamic_flag_count"]:
    if col not in flag_counts.columns:
        flag_counts[col] = 0
flag_counts["total_flag_count"] = flag_counts["static_flag_count"] + flag_counts["dynamic_flag_count"]
flag_counts = flag_counts.reset_index()

master_weekly = (active_students
    .merge(academic_weeks_df, on="academic_year", how="inner")
    .merge(who5_df[["student_id","academic_year","week_id","week_of_year","who5_total","delta_1w","rolling_mean_4w","persist_low"]],
           on=["student_id","academic_year","week_id","week_of_year"], how="left")
    .merge(engagement_df[["student_id","academic_year","week_id","total_engagement_score","logins","journal_entries","goals_set","no_engagement_2w"]],
           on=["student_id","academic_year","week_id"], how="left")
    .merge(flag_counts, on=["student_id","academic_year"], how="left")
)

display(master_weekly.head())
print(master_weekly.info())
print(master_weekly.describe())

master_weekly.to_csv(os.path.join(OUTPUT_DIR, "master_weekly.csv"), index=False)

# Yearly master
who5_yearly = who5_df.groupby(["student_id","academic_year"])["who5_total"].agg(["mean","std"]).reset_index().rename(
    columns={"mean":"who5_mean","std":"who5_sd"}
)
eng_yearly = engagement_df.groupby(["student_id","academic_year"])["total_engagement_score"].agg(["mean","std"]).reset_index().rename(
    columns={"mean":"engagement_mean","std":"engagement_sd"}
)
master_yearly = (students_df
    .merge(who5_yearly, on=["student_id","academic_year"], how="left")
    .merge(eng_yearly, on=["student_id","academic_year"], how="left")
    .merge(flag_counts, on=["student_id","academic_year"], how="left")
)

display(master_yearly.head())
print(master_yearly.info())
print(master_yearly.describe())

master_yearly.to_csv(os.path.join(OUTPUT_DIR, "master_yearly.csv"), index=False)


,student_id,academic_year,cohort,start_cohort,sub_cohort,status,week_id,week_of_year,week_start,who5_total,...,rolling_mean_4w,persist_low,total_engagement_score,logins,journal_entries,goals_set,no_engagement_2w,dynamic_flag_count,static_flag_count,total_flag_count
0,2af83920-568b-48fc-8d93-1bd1b1d54f4a,2019-2020,7,7,A,active,2019-2020-W01,1,2019-09-02,36,...,36.0,True,24.412429,1,1,3,False,1.0,0.0,1.0
1,2af83920-568b-48fc-8d93-1bd1b1d54f4a,2019-2020,7,7,A,active,2019-2020-W02,2,2019-09-09,48,...,42.0,True,50.876019,2,3,6,False,1.0,0.0,1.0
2,2af83920-568b-48fc-8d93-1bd1b1d54f4a,2019-2020,7,7,A,active,2019-2020-W03,3,2019-09-16,48,...,44.0,True,66.172879,4,3,9,False,1.0,0.0,1.0
3,2af83920-568b-48fc-8d93-1bd1b1d54f4a,2019-2020,7,7,A,active,2019-2020-W04,4,2019-09-23,40,...,43.0,True,53.854785,3,2,5,False,1.0,0.0,1.0
4,2af83920-568b-48fc-8d93-1bd1b1d54f4a,2019-2020,7,7,A,active,2019-2020-W05,5,2019-09-30,56,...,48.0,True,99.999998,5,4,10,False,1.0,0.0,1.0


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 36855 entries, 0 to 36854
Data columns (total 21 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   student_id              36855 non-null  object 
 1   academic_year           36855 non-null  object 
 2   cohort                  36855 non-null  int64  
 3   start_cohort            36855 non-null  int64  
 4   sub_cohort              36855 non-null  object 
 5   status                  36855 non-null  object 
 6   week_id                 36855 non-null  object 
 7   week_of_year            36855 non-null  int64  
 8   week_start              36855 non-null  object 
 9   who5_total              36855 non-null  int64  
 10  delta_1w                35910 non-null  float64
 11  rolling_mean_4w         36855 non-null  float64
 12  persist_low             36855 non-null  bool   
 13  total_engagement_score  36855 non-null  float64
 14  logins                  36855 non-null

,student_id,academic_year,cohort,start_cohort,sub_cohort,status,who5_mean,who5_sd,engagement_mean,engagement_sd,dynamic_flag_count,static_flag_count,total_flag_count
0,2af83920-568b-48fc-8d93-1bd1b1d54f4a,2019-2020,7,7,A,active,44.820513,14.427444,45.484832,22.150414,1.0,0.0,1.0
1,2af83920-568b-48fc-8d93-1bd1b1d54f4a,2020-2021,8,7,A,active,83.589744,9.917337,48.448291,23.985275,1.0,0.0,1.0
2,2af83920-568b-48fc-8d93-1bd1b1d54f4a,2021-2022,9,7,A,active,73.435897,9.558118,58.756647,24.042308,1.0,0.0,1.0
3,2af83920-568b-48fc-8d93-1bd1b1d54f4a,2022-2023,10,7,A,active,80.923077,12.327514,39.758263,24.581374,3.0,0.0,3.0
4,2af83920-568b-48fc-8d93-1bd1b1d54f4a,2023-2024,11,7,A,active,68.923077,14.464811,52.375165,26.735970,2.0,0.0,2.0


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 945 entries, 0 to 944
Data columns (total 13 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   student_id          945 non-null    object 
 1   academic_year       945 non-null    object 
 2   cohort              945 non-null    int64  
 3   start_cohort        945 non-null    int64  
 4   sub_cohort          945 non-null    object 
 5   status              945 non-null    object 
 6   who5_mean           945 non-null    float64
 7   who5_sd             945 non-null    float64
 8   engagement_mean     945 non-null    float64
 9   engagement_sd       945 non-null    float64
 10  dynamic_flag_count  885 non-null    float64
 11  static_flag_count   885 non-null    float64
 12  total_flag_count    885 non-null    float64
dtypes: float64(7), int64(2), object(4)
memory usage: 96.1+ KB
None
           cohort  start_cohort   who5_mean     who5_sd  engagement_mean  \
count  945.00000


## 8) EDA Columns

We compute per **student-year**:
- **WHO-5 skewness & kurtosis**
- **WHO-5 volatility** (`who5_volatility = SD` of weekly totals)
- **Engagement volatility** (`engagement_volatility = SD` of weekly scores)
- **Within-year correlation** between weekly WHO-5 and engagement
- **Flag prevalence rates** per cohort-year (share of students with ≥1 dynamic flag)

Outputs:
- `eda_student_year_df.csv`
- `eda_cohort_year_df.csv` (cohort-level prevalence)


In [16]:

# Per student-year time series
def eda_metrics_for_group(g):
    who = g["who5_total"].dropna()
    eng = g["total_engagement_score"].dropna()
    metrics = {}
    if len(who) > 1:
        metrics["who5_skew"] = float(skew(who, bias=False))
        metrics["who5_kurtosis"] = float(kurtosis(who, bias=False))
        metrics["who5_volatility"] = float(np.std(who, ddof=1))
    else:
        metrics["who5_skew"] = np.nan
        metrics["who5_kurtosis"] = np.nan
        metrics["who5_volatility"] = np.nan
    if len(eng) > 1:
        metrics["engagement_volatility"] = float(np.std(eng, ddof=1))
    else:
        metrics["engagement_volatility"] = np.nan
    # correlation
    if len(who)==len(eng) and len(who) > 2:
        try:
            r,_ = pearsonr(who, eng)
        except Exception:
            r = np.nan
    else:
        # align by index if needed
        try:
            aligned = g[["who5_total","total_engagement_score"]].dropna()
            if len(aligned) > 2:
                r,_ = pearsonr(aligned["who5_total"], aligned["total_engagement_score"])
            else:
                r = np.nan
        except Exception:
            r = np.nan
    metrics["who5_eng_corr"] = float(r) if r == r else np.nan
    return pd.Series(metrics)

mw = master_weekly.copy()
eda_sy = (mw.groupby(["student_id","academic_year"]).apply(eda_metrics_for_group).reset_index())
eda_student_year_df = (students_df
                       .merge(eda_sy, on=["student_id","academic_year"], how="left"))

display(eda_student_year_df.head())
print(eda_student_year_df.info())
print(eda_student_year_df.describe())

eda_student_year_df.to_csv(os.path.join(OUTPUT_DIR, "eda_student_year_df.csv"), index=False)

# Cohort-year prevalence of dynamic flags (≥1 per student-year)
dyn_any = safeguarding_flags_df[safeguarding_flags_df["flag_type"]=="dynamic"].groupby(
    ["student_id","academic_year"]
)["flag_name"].nunique().reset_index(name="dyn_flag_count")
dyn_any["has_dynamic_flag"] = dyn_any["dyn_flag_count"] > 0

cohort_map = students_df[["student_id","academic_year","cohort"]].copy()
cohort_dyn = cohort_map.merge(dyn_any[["student_id","academic_year","has_dynamic_flag"]],
                              on=["student_id","academic_year"], how="left").fillna(False)
prevalence = (cohort_dyn.groupby(["academic_year","cohort"])["has_dynamic_flag"]
              .mean().reset_index().rename(columns={"has_dynamic_flag":"dynamic_flag_prevalence"}))

display(prevalence.head())
print(prevalence.info())
print(prevalence.describe())

prevalence.to_csv(os.path.join(OUTPUT_DIR, "eda_cohort_year_df.csv"), index=False)


/tmp/ipython-input-1129983371.py:38: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  eda_sy = (mw.groupby(["student_id","academic_year"]).apply(eda_metrics_for_group).reset_index())


,student_id,academic_year,cohort,start_cohort,sub_cohort,status,who5_skew,who5_kurtosis,who5_volatility,engagement_volatility,who5_eng_corr
0,2af83920-568b-48fc-8d93-1bd1b1d54f4a,2019-2020,7,7,A,active,0.360266,-0.358428,14.427444,22.150414,0.539403
1,2af83920-568b-48fc-8d93-1bd1b1d54f4a,2020-2021,8,7,A,active,-0.258595,-0.860023,9.917337,23.985275,0.541633
2,2af83920-568b-48fc-8d93-1bd1b1d54f4a,2021-2022,9,7,A,active,-0.133099,-0.889286,9.558118,24.042308,0.589266
3,2af83920-568b-48fc-8d93-1bd1b1d54f4a,2022-2023,10,7,A,active,-1.644446,5.296185,12.327514,24.581374,0.371237
4,2af83920-568b-48fc-8d93-1bd1b1d54f4a,2023-2024,11,7,A,active,0.285365,-0.380290,14.464811,26.735970,0.534833


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 945 entries, 0 to 944
Data columns (total 11 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   student_id             945 non-null    object 
 1   academic_year          945 non-null    object 
 2   cohort                 945 non-null    int64  
 3   start_cohort           945 non-null    int64  
 4   sub_cohort             945 non-null    object 
 5   status                 945 non-null    object 
 6   who5_skew              945 non-null    float64
 7   who5_kurtosis          945 non-null    float64
 8   who5_volatility        945 non-null    float64
 9   engagement_volatility  945 non-null    float64
 10  who5_eng_corr          945 non-null    float64
dtypes: float64(5), int64(2), object(4)
memory usage: 81.3+ KB
None
           cohort  start_cohort   who5_skew  who5_kurtosis  who5_volatility  \
count  945.000000         945.0  945.000000     945.000000       945.000

/tmp/ipython-input-1129983371.py:56: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  on=["student_id","academic_year"], how="left").fillna(False)


,academic_year,cohort,dynamic_flag_prevalence
0,2019-2020,7,0.777778
1,2020-2021,7,0.866667
2,2020-2021,8,0.777778
3,2021-2022,7,0.888889
4,2021-2022,8,0.911111


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 21 entries, 0 to 20
Data columns (total 3 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   academic_year            21 non-null     object 
 1   cohort                   21 non-null     int64  
 2   dynamic_flag_prevalence  21 non-null     float64
dtypes: float64(1), int64(1), object(1)
memory usage: 636.0+ bytes
None
          cohort  dynamic_flag_prevalence
count  21.000000                21.000000
mean    8.666667                 0.844444
std     1.527525                 0.055777
min     7.000000                 0.733333
25%     7.000000                 0.822222
50%     8.000000                 0.844444
75%    10.000000                 0.888889
max    12.000000                 0.933333



## 9) Save All CSVs & Quick Summary

All tables have been saved to `/content/data/`. Below we list file names and basic shapes.


In [19]:
import glob, os
files = sorted(glob.glob(os.path.join(OUTPUT_DIR, "*.csv")))
summary = []
for f in files:
    try:
        df = pd.read_csv(f)  # Read the full CSV
        summary.append({"file": os.path.basename(f),
                        "columns": df.shape[1],
                        "rows": df.shape[0],
                        "features": df.columns.tolist()}) # Add the list of column names
    except Exception as e:
        summary.append({"file": os.path.basename(f), "columns": "?", "rows": "?", "features": "?"})
pd.DataFrame(summary)

,file,columns,rows,features
0,academic_weeks_df.csv,4,234,"[week_id, academic_year, week_of_year, week_st..."
1,academic_years_df.csv,2,6,"[academic_year, year_start]"
2,eda_cohort_year_df.csv,3,21,"[academic_year, cohort, dynamic_flag_prevalence]"
3,eda_student_year_df.csv,11,945,"[student_id, academic_year, cohort, start_coho..."
4,engagement_df.csv,10,36855,"[student_id, academic_year, week_id, week_of_y..."
5,master_weekly.csv,21,36855,"[student_id, academic_year, cohort, start_coho..."
6,master_yearly.csv,13,945,"[student_id, academic_year, cohort, start_coho..."
7,safeguarding_flags_df.csv,4,2526,"[student_id, academic_year, flag_name, flag_type]"
8,students_df.csv,6,945,"[student_id, academic_year, cohort, start_coho..."
9,term_days_df.csv,5,1170,"[week_id, academic_year, week_of_year, date, d..."


In [ ]:
!pip install GitPython --quiet

In [22]:
import git
import os
import shutil

# Define repository details and local paths
repo_url = "https://github.com/phoenixcapera04/Dissertation-R-Shiny/"
repo_name = "Dissertation-R-Shiny" # Update repo_name to match the new URL
repo_dir = os.path.join("/content", repo_name)
data_source_dir = "/content/data"
data_target_dir = os.path.join(repo_dir, "data") # Assuming a 'data' directory in your repo

# Clone the repository
if os.path.exists(repo_dir):
    print(f"Repository already exists at {repo_dir}. Pulling latest changes.")
    repo = git.Repo(repo_dir)
    repo.remotes.origin.pull()
else:
    print(f"Cloning repository {repo_url} to {repo_dir}")
    repo = git.Repo.clone_from(repo_url, repo_dir)

# Ensure the target data directory exists in the repository
os.makedirs(data_target_dir, exist_ok=True)

# Copy generated CSV files to the repository's data directory
print(f"Copying CSV files from {data_source_dir} to {data_target_dir}")
for filename in os.listdir(data_source_dir):
    if filename.endswith(".csv"):
        shutil.copy(os.path.join(data_source_dir, filename), data_target_dir)
        print(f"Copied {filename}")

# Add changes to the repository
print("Adding changes to git")
repo.git.add(data_target_dir)

# Commit changes
commit_message = "Add generated synthetic data CSVs"
print(f"Committing with message: '{commit_message}'")
try:
    repo.git.commit("-m", commit_message)
    print("Commit successful")
except git.exc.GitCommandError as e:
    if "nothing to commit" in str(e):
        print("No changes to commit.")
    else:
        print(f"Error during commit: {e}")

# Push changes
print(f"Pushing changes to origin/{repo.active_branch.name}")
try:
    # Use --force-with-lease or handle potential authentication issues here
    # For simplicity, assuming credentials are set up or using Colab's mechanism
    repo.remotes.origin.push()
    print("Push successful!")
except git.exc.GitCommandError as e:
    print(f"Error during push: {e}")
    print("Please ensure you have configured your Git credentials or have a valid PAT.")

Cloning repository https://github.com/phoenixcapera04/Dissertation-R-Shiny/ to /content/Dissertation-R-Shiny
Copying CSV files from /content/data to /content/Dissertation-R-Shiny/data
Copied eda_cohort_year_df.csv
Copied academic_years_df.csv
Copied term_days_df.csv
Copied engagement_df.csv
Copied master_weekly.csv
Copied safeguarding_flags_df.csv
Copied academic_weeks_df.csv
Copied who5_df.csv
Copied eda_student_year_df.csv
Copied master_yearly.csv
Copied students_df.csv
Adding changes to git
Committing with message: 'Add generated synthetic data CSVs'
Error during commit: Cmd('git') failed due to: exit code(128)
  cmdline: git commit -m Add generated synthetic data CSVs
  stderr: 'Author identity unknown

*** Please tell me who you are.

Run

  git config --global user.email "you@example.com"
  git config --global user.name "Your Name"

to set your account's default identity.
Omit --global to set the identity only in this repository.

fatal: unable to auto-detect email address (got '